In [1]:
from eyened_orm import Database
from dotenv import load_dotenv

load_dotenv("/home/bart/.env.dev")
session = Database().create_session()

In [2]:
from eyened_orm import ImageInstance


def print_attribute_value(av):
    print(f"{av.AttributeDefinition.AttributeName} ({av.AttributeValueID})")
    print("  model version:", av.ProducingModel.Version)
    print("  value:", av.value)

    if av.InputValues:
        print("  provenance tracking:")
        for inp in av.InputValues:
            print(
                f"  └─ {inp.AttributeValueID} ({inp.AttributeDefinition.AttributeName})"
            )


def refresh_image(image_id):
    session.rollback()
    return ImageInstance.by_id(session, image_id)

In [3]:
image_id = 331115

# remove any existing attribute values
im = refresh_image(image_id)
for av in list(im.AttributeValues):
    session.delete(av)
session.commit()

print("before run-cfi-models:")
for av in im.AttributeValues:
    print_attribute_value(av)
    print()

before run-cfi-models:


# Run all models
`eorm run-cfi-models`

In [4]:
# filter by image_id (optional)
!eorm --env-file /home/bart/.env.dev run-cfi-models --image-ids={image_id}

Connected to database eyened_database on eyened-supergpu:16306
Target: 1 from --image-ids
Running cfi-roi
Processing 1 images (after filtering, default)
100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  5.12it/s]
Completed processing 1 images
Running cfi-keypoints
Processing 1 images (after filtering, default)
  0%|                                                     | 0/1 [00:00<?, ?it/s]loading fovea models
loading discedge models
/home/bart/.venvs/default/lib/python3.12/site-packages/monai/inferers/utils.py:231: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
  win_data = inputs[unravel_slice[0]].to(sw_device)
/home/bart/.venvs/defa

In [5]:
# show all attributes plus provenance tracking
im = refresh_image(image_id)
for av in im.AttributeValues:
    print_attribute_value(av)
    print()

CFI_Quality (4364170)
  model version: Eyened/vascx/quality/quality
  value: 1.65316
  provenance tracking:
  └─ 4364167 (CFI_ROI)

CFI_ROI (4364167)
  model version: 1.1.0
  value: {'hw': [1632, 2464], 'lines': {}, 'max_x': 2464, 'max_y': 1632, 'min_x': 0, 'min_y': 0, 'center': [1225.2965427136887, 812.1591129771098], 'radius': 803.707105254458}

CFI_Keypoints (4364168)
  model version: Eyened/vascx/discedge/discedge_july24+Eyened/vascx/fovea/fovea_july24
  value: {'fovea_xy': [1223.7268022737387, 838.8447004562618], 'disc_edge_xy': [1839.0650547341831, 813.72885341706]}
  provenance tracking:
  └─ 4364167 (CFI_ROI)

CFI_ODFD (4364169)
  model version: Eyened/vascx/odfd/odfd_march25
  value: 542.577
  provenance tracking:
  └─ 4364167 (CFI_ROI)



In [6]:
# generic accessor for any attribute
# pass in the attribute_name (AttributeDefinition.AttributeName)
im.get_attribute_value(attribute_name="CFI_Quality")

1.65316

In [7]:
# shortcut properties are available for cfi-models attributes
display(im.cfi_quality)
display(im.cfi_keypoints)
display(im.cfi_roi)
display(im.cfi_odfd)

1.65316

{'fovea_xy': [1223.7268022737387, 838.8447004562618],
 'disc_edge_xy': [1839.0650547341831, 813.72885341706]}

{'hw': [1632, 2464],
 'lines': {},
 'max_x': 2464,
 'max_y': 1632,
 'min_x': 0,
 'min_y': 0,
 'center': [1225.2965427136887, 812.1591129771098],
 'radius': 803.707105254458}

542.577

In [8]:
# multiple model versions: reads pick the highest version
from eyened_orm import AttributeDefinition, AttributeValue, AttributesModel

# adding a legacy model for demo purposes
legacy_version = "Eyened/vascx/discedge/discedge_july23+Eyened/vascx/fovea/fovea_july23"
attr = AttributeDefinition.by_column(session, AttributeName="CFI_Keypoints")
legacy_model = AttributesModel.get_or_create(
    session,
    match_by={"ModelName": "CFI_Keypoints", "Version": legacy_version},
    update_values={"Description": "legacy demo"},
)
# adding a value for the legacy model
AttributeValue.upsert(
    session,
    match_by={
        "AttributeID": attr.AttributeID,
        "ModelID": legacy_model.ModelID,
        "ImageInstanceID": image_id,
    },
    update_values={
        "ValueJSON": {"fovea_xy": [0.0, 0.0], "disc_edge_xy": [0.0, 0.0]},
    },
)
session.commit()

In [9]:
im = refresh_image(image_id)
# shows multiple outputs for the same attribute
print("stored rows:")
for av in im.AttributeValues:
    if av.AttributeDefinition.AttributeName == "CFI_Keypoints":
        print_attribute_value(av)
        print()

# automatically picks the highest version
print("cfi_keypoints property:", im.cfi_keypoints)

stored rows:
CFI_Keypoints (4364168)
  model version: Eyened/vascx/discedge/discedge_july24+Eyened/vascx/fovea/fovea_july24
  value: {'fovea_xy': [1223.7268022737387, 838.8447004562618], 'disc_edge_xy': [1839.0650547341831, 813.72885341706]}
  provenance tracking:
  └─ 4364167 (CFI_ROI)

CFI_Keypoints (4364171)
  model version: Eyened/vascx/discedge/discedge_july23+Eyened/vascx/fovea/fovea_july23
  value: {'fovea_xy': [0.0, 0.0], 'disc_edge_xy': [0.0, 0.0]}

cfi_keypoints property: {'fovea_xy': [1223.7268022737387, 838.8447004562618], 'disc_edge_xy': [1839.0650547341831, 813.72885341706]}


## Upgrade
`--upgrade`: run current version when only an older version exists 

In [10]:
from eyened_orm.inference.cfi_keypoints import CFIKeypoints

## removing existing current version for demo purposes
im = refresh_image(image_id)
current_version = CFIKeypoints(session, device=None).model_version
for av in list(im.AttributeValues):
    if (
        av.AttributeDefinition.AttributeName == "CFI_Keypoints"
        and av.ProducingModel.Version == current_version
    ):
        session.delete(av)
session.commit()

In [11]:
print("before upgrade (legacy keypoints only):")
for av in refresh_image(image_id).AttributeValues:
    if av.AttributeDefinition.AttributeName == "CFI_Keypoints":
        print_attribute_value(av)

before upgrade (legacy keypoints only):
CFI_Keypoints (4364171)
  model version: Eyened/vascx/discedge/discedge_july23+Eyened/vascx/fovea/fovea_july23
  value: {'fovea_xy': [0.0, 0.0], 'disc_edge_xy': [0.0, 0.0]}


In [12]:
## run without upgrade maintains legacy version (not processed again)
!eorm --env-file /home/bart/.env.dev run-cfi-models -m cfi-keypoints --image-ids={image_id}

Connected to database eyened_database on eyened-supergpu:16306
Target: 1 from --image-ids
Running cfi-keypoints
Skipping 1 images with existing results
Processing 0 images (after filtering, default)
No images to process


In [13]:
## run with upgrade to process the image for newer model versions
!eorm --env-file /home/bart/.env.dev run-cfi-models --upgrade -m cfi-keypoints --image-ids={image_id}

Connected to database eyened_database on eyened-supergpu:16306
Target: 1 from --image-ids
Running cfi-keypoints
Processing 1 images (after filtering, upgrade)
  0%|                                                     | 0/1 [00:00<?, ?it/s]loading fovea models
loading discedge models
Using local storage
/home/bart/.venvs/default/lib/python3.12/site-packages/monai/inferers/utils.py:231: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
  win_data = inputs[unravel_slice[0]].to(sw_device)
/home/bart/.venvs/default/lib/python3.12/site-packages/monai/inferers/utils.py:370: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will

In [14]:
im = refresh_image(image_id)
print("after upgrade (both versions stored):")
for av in im.AttributeValues:
    if av.AttributeDefinition.AttributeName == "CFI_Keypoints":
        print_attribute_value(av)
        print()

# automatically picks the highest version
print("cfi_keypoints property:", im.cfi_keypoints)

after upgrade (both versions stored):
CFI_Keypoints (4364172)
  model version: Eyened/vascx/discedge/discedge_july24+Eyened/vascx/fovea/fovea_july24
  value: {'fovea_xy': [1223.7268022737387, 838.8447004562618], 'disc_edge_xy': [1839.0650547341831, 813.72885341706]}
  provenance tracking:
  └─ 4364167 (CFI_ROI)

CFI_Keypoints (4364171)
  model version: Eyened/vascx/discedge/discedge_july23+Eyened/vascx/fovea/fovea_july23
  value: {'fovea_xy': [0.0, 0.0], 'disc_edge_xy': [0.0, 0.0]}

cfi_keypoints property: {'fovea_xy': [1223.7268022737387, 838.8447004562618], 'disc_edge_xy': [1839.0650547341831, 813.72885341706]}


## Failed
`--failed`: retry rows with null value columns

In [15]:
from eyened_orm.inference.cfi_odfd import CFI_ODFD

# removing existing current version and add a failed row
# NOTE: this is equivalent to setting the value to NULL in the database
im = refresh_image(image_id)
for av in list(im.AttributeValues):
    if av.AttributeDefinition.AttributeName == "CFI_ODFD":
        session.delete(av)
session.flush()

# add a failed row
CFI_ODFD(session, device=None)._save_failure(image_id)
session.commit()

# shows the failed row
print("simulated failed CFI_ODFD row:")
for av in refresh_image(image_id).AttributeValues:
    if av.AttributeDefinition.AttributeName == "CFI_ODFD":
        print_attribute_value(av)

simulated failed CFI_ODFD row:
CFI_ODFD (4364173)
  model version: Eyened/vascx/odfd/odfd_march25
  value: None


In [16]:
# default mode (without `--failed`) skips failed images
!eorm --env-file /home/bart/.env.dev run-cfi-models -m cfi-odfd --image-ids={image_id}

Connected to database eyened_database on eyened-supergpu:16306
Target: 1 from --image-ids
Running cfi-odfd
Skipping 1 images with existing results
Processing 0 images (after filtering, default)
No images to process


In [17]:
# with `--failed` only failed images in the target set are processed
!eorm --env-file /home/bart/.env.dev run-cfi-models --failed -m cfi-odfd --image-ids={image_id}

Connected to database eyened_database on eyened-supergpu:16306
Target: 1 from --image-ids
Running cfi-odfd
Scoped to 1 images with failed cfi-odfd output
Processing 1 images (after filtering, failed)
100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.59s/it]
Completed processing 1 images


In [18]:
im = refresh_image(image_id)
for av in im.AttributeValues:
    if av.AttributeDefinition.AttributeName == "CFI_ODFD":
        print_attribute_value(av)

CFI_ODFD (4364173)
  model version: Eyened/vascx/odfd/odfd_march25
  value: 542.577
  provenance tracking:
  └─ 4364167 (CFI_ROI)


## Overwrite
`--overwrite`: re-run current version even when output exists

In [19]:
# simulate alternative value in database
av = im.find_attribute_value(attribute_name="CFI_Quality")
av.ValueFloat = -1.0
session.add(av)
session.commit()

im = refresh_image(image_id)
print("quality before:", im.cfi_quality)

quality before: -1.0


In [20]:
# without `--overwrite`: skips images with existing output
!eorm --env-file /home/bart/.env.dev run-cfi-models -m cfi-quality --image-ids={image_id}

Connected to database eyened_database on eyened-supergpu:16306
Target: 1 from --image-ids
Running cfi-quality
Skipping 1 images with existing results
Processing 0 images (after filtering, default)
No images to process


In [21]:
# with `--overwrite`: re-run current version even when output already exists (updates the value)
!eorm --env-file /home/bart/.env.dev run-cfi-models --overwrite -m cfi-quality --image-ids={image_id}

Connected to database eyened_database on eyened-supergpu:16306
Target: 1 from --image-ids
Running cfi-quality
Processing 1 images (overwrite)
100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.57s/it]
Completed processing 1 images


In [22]:
im = refresh_image(image_id)
print("quality after:", im.cfi_quality)
for av in im.AttributeValues:
    if av.AttributeDefinition.AttributeName == "CFI_Quality":
        print_attribute_value(av)

quality after: 1.65316
CFI_Quality (4364170)
  model version: Eyened/vascx/quality/quality
  value: 1.65316
  provenance tracking:
  └─ 4364167 (CFI_ROI)
